# Codificación de preguntas abiertas con Qwen3

Paso del medio de la codificación, con la misma forma que el notebook de Whisper:

1. R deja en **`Análisis/C/<ronda>`** dos archivos: `prompts_<ronda>.csv` y `tareas_<ronda>.csv`.
2. **Este notebook** corre el modelo y escribe `codigos_<ronda>.csv` en la misma carpeta.
3. R lo baja y lo pega a la base de la ronda.

Toda la lógica (codebook, dependencias, prompts) está resuelta en R: acá solo se
ejecuta el modelo. El notebook no sabe nada del codebook más allá de la lista de
etiquetas válidas que viene en `prompts_<ronda>.csv`.

**Es reanudable:** guarda resultados parciales, así que si se corta se lo vuelve a
correr y sigue con lo que falta.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

In [ ]:
#@title 1) Instalar Ollama y levantar el servidor
# Se usa el mismo motor que en local, asi los resultados son comparables.
import os, shutil, subprocess, time, requests

os.environ['OLLAMA_NUM_PARALLEL'] = '4'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '4096'

def sh(cmd):
    """Corre un comando mostrando su salida (nada de esconder errores)."""
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    salida = (r.stdout + r.stderr).strip()
    if salida: print(salida[-1500:])
    return r.returncode

if shutil.which('ollama') is None:
    # el instalador de Ollama descomprime con zstd y la imagen de Colab no lo trae
    if shutil.which('zstd') is None:
        sh('apt-get -qq update && apt-get -qq install -y zstd')
    sh('curl -fsSL https://ollama.com/install.sh | sh')

if shutil.which('ollama') is None:
    # si quedo una instalacion a medias, limpiarla y reintentar una vez
    sh('rm -rf /usr/local/lib/ollama')
    sh('curl -fsSL https://ollama.com/install.sh | sh')

if shutil.which('ollama') is None:
    raise SystemExit('no se pudo instalar ollama: revisar la salida de arriba')

print('binario:', shutil.which('ollama'))

LOG = '/content/ollama.log'
servidor = subprocess.Popen(['ollama', 'serve'],
                            stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)

BASE = 'http://127.0.0.1:11434'
for _ in range(60):
    try:
        requests.get(f'{BASE}/api/tags', timeout=2)
        print('ollama responde en', BASE)
        break
    except Exception:
        time.sleep(1)
else:
    print(open(LOG).read()[-2000:])
    raise SystemExit('ollama no levanto: ver el log de arriba')

In [ ]:
#@title 2) Descargar el modelo
MODELO = "qwen3:8b"  #@param ["qwen3:8b", "qwen3:14b", "qwen3:4b"]

if sh(f'ollama pull {MODELO}') != 0:
    raise SystemExit(f'no se pudo descargar {MODELO}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

In [ ]:
#@title 3) Montar Drive y encontrar la carpeta de la ronda
from google.colab import drive
drive.mount('/content/drive')

ROUND_ID = "R1"          #@param {type:"string"}
NOMBRE_RAIZ = "Análisis" #@param {type:"string"}
RUTA_MANUAL = ""         #@param {type:"string"}

import glob
from pathlib import Path

def buscar_raiz(nombre):
    patrones = [f"/content/drive/MyDrive/{nombre}",
                f"/content/drive/MyDrive/*/{nombre}",
                f"/content/drive/MyDrive/*/*/{nombre}",
                f"/content/drive/MyDrive/*/*/*/{nombre}",
                f"/content/drive/.shortcut-targets-by-id/*/{nombre}",
                f"/content/drive/.shortcut-targets-by-id/*/*/{nombre}"]
    return sorted({c for p in patrones for c in glob.glob(p)
                   if os.path.isdir(os.path.join(c, 'C'))})

if RUTA_MANUAL:
    raiz = Path(RUTA_MANUAL)
else:
    encontradas = buscar_raiz(NOMBRE_RAIZ)
    if not encontradas:
        raise SystemExit(
            f"No encontré '{NOMBRE_RAIZ}' con la subcarpeta C dentro de Mi unidad.\n"
            "Si la carpeta del equipo es compartida, agregale un acceso directo a Mi unidad\n"
            "(click derecho sobre la CARPETA → Organizar → Añadir acceso directo).")
    raiz = Path(encontradas[0])

dir_ronda = raiz / 'C' / ROUND_ID
assert dir_ronda.exists(), f"No existe {dir_ronda}: correr primero DriveFlow/exportar_para_colab.R"
print('carpeta:', dir_ronda)
print(sorted(p.name for p in dir_ronda.glob('*.csv')))

In [ ]:
#@title 4) Leer las tareas y los prompts que dejó R
import pandas as pd

f_tareas  = dir_ronda / f'tareas_{ROUND_ID}.csv'
f_prompts = dir_ronda / f'prompts_{ROUND_ID}.csv'
f_salida  = dir_ronda / f'codigos_{ROUND_ID}.csv'

tareas  = pd.read_csv(f_tareas,  dtype={'fila': int, 'pregunta': str, 'grupo': str, 'texto': str})
prompts = pd.read_csv(f_prompts, dtype=str)

# un prompt y una lista de etiquetas por (pregunta, grupo)
PROMPTS = {(r.pregunta, r.grupo): (r.prompt_sistema, r.etiquetas.split('|'))
           for r in prompts.itertuples()}

faltan = {(r.pregunta, r.grupo) for r in tareas.itertuples()} - set(PROMPTS)
assert not faltan, f"tareas sin prompt: {faltan}"

print(f'{len(tareas)} respuestas a codificar | {len(PROMPTS)} prompts')
print(tareas.groupby('pregunta').size().to_string())

In [ ]:
#@title 5) Codificar (reanudable: se puede volver a correr esta celda sola)
import json, requests, time

MAX_CODIGOS = 3
GUARDAR_CADA = 25   # cada cuántas respuestas se escribe el parcial en Drive

PARCIAL = Path('/content/parcial.csv')

def hechos_previos():
    """Lo ya codificado, del parcial local o de la salida en Drive."""
    for f in (PARCIAL, f_salida):
        if f.exists():
            d = pd.read_csv(f, dtype={'fila': int, 'pregunta': str, 'codigos': str})
            d = d[d.codigos != 'ERROR']          # los errores se reintentan
            return d
    return pd.DataFrame(columns=['fila', 'pregunta', 'codigos'])

def esquema(etiquetas):
    """JSON Schema que obliga al modelo a elegir del codebook y nada más."""
    return {"type": "object",
            "properties": {"codigos": {"type": "array",
                                       "items": {"type": "string", "enum": etiquetas},
                                       "maxItems": MAX_CODIGOS}},
            "required": ["codigos"]}

def clasificar(prompt_sistema, etiquetas, texto):
    r = requests.post(f'{BASE}/api/chat', timeout=180, json={
        'model': MODELO,
        'messages': [{'role': 'system', 'content': prompt_sistema},
                     {'role': 'user',   'content': texto}],
        'format': esquema(etiquetas),
        'think': False,
        'stream': False,
        'options': {'temperature': 0, 'seed': 1234, 'num_predict': 128},
    })
    r.raise_for_status()
    codigos = json.loads(r.json()['message']['content']).get('codigos', [])
    codigos = [c for c in dict.fromkeys(codigos) if c in etiquetas][:MAX_CODIGOS]
    return '; '.join(codigos) if codigos else 'ERROR'

hechos = hechos_previos()
ya = set(zip(hechos.fila, hechos.pregunta))
resultados = hechos.to_dict('records')
pendientes = [t for t in tareas.itertuples() if (t.fila, t.pregunta) not in ya]

print(f'ya codificadas: {len(ya)} | pendientes: {len(pendientes)}')
errores, t0 = [], time.time()

for i, t in enumerate(pendientes, 1):
    prompt_sistema, etiquetas = PROMPTS[(t.pregunta, t.grupo)]
    try:
        codigos = clasificar(prompt_sistema, etiquetas, t.texto)
    except Exception as e:
        codigos = 'ERROR'
        errores.append(f'{t.pregunta} fila {t.fila}: {type(e).__name__}: {e}')

    resultados.append({'fila': t.fila, 'pregunta': t.pregunta, 'codigos': codigos})

    if i % GUARDAR_CADA == 0 or i == len(pendientes):
        df = pd.DataFrame(resultados)
        df.to_csv(PARCIAL, index=False)
        df.to_csv(f_salida, index=False)
        hechas = (time.time() - t0) / i
        print(f'[{i}/{len(pendientes)}] {hechas:.1f}s por respuesta · '
              f'faltan ~{(len(pendientes)-i)*hechas/60:.0f} min')

print(f'\nlisto en {(time.time()-t0)/60:.1f} min | errores: {len(errores)}')
for e in errores[:10]: print('  -', e)

In [ ]:
#@title 6) Control final
final = pd.read_csv(f_salida, dtype={'fila': int, 'pregunta': str, 'codigos': str})

print(f'{len(final)} de {len(tareas)} respuestas codificadas')
print(f"errores: {(final.codigos == 'ERROR').sum()}")

for q, sub in final.groupby('pregunta'):
    print(f'\n--- {q} ---')
    cods = sub.codigos[sub.codigos != 'ERROR'].str.split('; ').explode()
    print(cods.value_counts().head(8).to_string())

print(f'\nGuardado en {f_salida}')
print('Volver a R y correr DriveFlow/importar_de_colab.R')